# Lesson 04 — A validation loop (your first cycle)

```
START -> extract -> validate -> (loop back to extract | END)
```

The extractor is only trustworthy if every cited `excerpt` actually appears in
the transcript. So we add a `validate` node that checks exactly that, and a
**conditional edge** that loops back to `extract` — feeding the hallucinated
excerpts back to the model — until the citations are clean or we hit
`MAX_ATTEMPTS`.

**LangGraph concepts introduced**
- a **cycle**: an edge that points *backwards* (the reason to use a graph, not a chain)
- **conditional edges** via `add_conditional_edges(node, router, {...})`
- **state carried across supersteps**: `attempts` and `errors`

**Design decision:** we require excerpts to be *verbatim* substrings (see the
excerpt field in `clinical/schema.py`). That makes validation a simple, reliable
string check instead of fuzzy matching — and makes the loop observable.

The graph itself lives in `graphs/validation_loop.py` so LangGraph Studio can load
it too (see the bottom of this notebook).

In [ ]:
from clinical import (
    Evidence, EvidenceBackedText, SymptomProblem, SymptomExtraction,
    find_unsupported_excerpts, show_symptom_evidence_matrix,
    get_transcript,
)
from graphs.validation_loop import build_graph, MAX_ATTEMPTS

## 1. The validator, on its own (no LLM)

`find_unsupported_excerpts` is plain Python. Prove it works before wiring it into
a graph: a made-up excerpt is flagged, a verbatim one is not.

In [ ]:
transcript = "[00:00] Patsient: valu on parema alakohu piirkonnas. [00:12] Arst: kas kiirgub?"

good = SymptomExtraction(problems=[SymptomProblem(
    problem="parema alakohu valu",
    core_evidence=[Evidence(speaker="patient", timestamp="00:00",
                            excerpt="valu on parema alakohu piirkonnas")],
)])
bad = SymptomExtraction(problems=[SymptomProblem(
    problem="parema alakohu valu",
    core_evidence=[Evidence(speaker="patient", timestamp="00:00",
                            excerpt="valu vasakus ola piirkonnas")],  # never said
)])

print("good ->", find_unsupported_excerpts(good, transcript))   # []
print("bad  ->", find_unsupported_excerpts(bad, transcript))    # one flagged excerpt

## 2. Watch the loop run — with a stub model (still no LLM, no cost)

To *see the cycle* deterministically, inject a fake model that hallucinates on its
first call and fixes itself on the second. The graph should run `extract` twice
and then end clean. This is why `build_graph(model=...)` takes the model as an
argument.

In [ ]:
class StubModel:
    # Returns a bad extraction first, then a good one — to force exactly one loop.
    def __init__(self, transcript):
        self.transcript = transcript
        self.calls = 0

    def invoke(self, messages):
        self.calls += 1
        if self.calls == 1:
            return bad      # hallucinated excerpt -> validate fails -> loop
        return good         # verbatim excerpt -> validate passes -> END


stub = StubModel(transcript)
demo_graph = build_graph(model=stub)
final = demo_graph.invoke({"transcript": transcript, "encounter_date": "27.11.2025"})

print("model was called", stub.calls, "times")
print("attempts recorded in state:", final["attempts"])
print("remaining errors:", final["errors"])

In [ ]:
# The cycle, drawn. Note the edge from `validate` back to `extract`.
print(demo_graph.get_graph().draw_mermaid())

## 3. The real thing

Same graph, real model. Needs `OPENROUTER_API_KEY` in `.env` and costs up to
`MAX_ATTEMPTS` calls. Watch `attempts` — if it's > 1, the model hallucinated a
citation and the loop repaired it.

In [ ]:
graph = build_graph()  # uses clinical.get_extraction_model()
result = graph.invoke({
    "transcript": get_transcript("transcript_1"),
    "encounter_date": "27.11.2025",
})
print(f"attempts={result['attempts']}  unresolved_errors={len(result['errors'])}  (max={MAX_ATTEMPTS})")
show_symptom_evidence_matrix(result["extraction"])

## Try it yourself

- Set `build_graph(max_attempts=1)` — the loop can't repair; compare `errors`.
- Loosen the excerpt rule in `clinical/prompts.py` back to "preserve meaning" and
  watch attempts climb (paraphrases fail the verbatim check).
- Stream the run: `for step in graph.stream({...}, stream_mode="updates"): print(step)`
  — a preview of lesson 09.

## See it in LangGraph Studio

From the repo root:

```bash
langgraph dev
```

Studio loads `validation_loop` from `langgraph.json`, and you can step through the
`extract -> validate -> extract` loop visually — the clearest way to *watch* a
cycle execute.